# Routing Queries Within One Large PDF Using Chunk Metadata

**Document:** `pharma-blob-sample.pdf` (10 pages, 8 bundled documents)
**Stack:** LangChain `RecursiveCharacterTextSplitter` + LlamaIndex (BGE-small embeddings, metadata filters, response synthesizer) + **Mistral-7B-Instruct** (local, llama.cpp) for query routing and answers
**Author:** Bashar

Within a single file, the pipeline should:

1. tag every chunk with the document it came from (`doc_type`, `doc_id`, `page`, `page_in_doc`, `chunk_index`);
2. predict which document type a question is about;
3. retrieve only chunks of that type;
4. write an answer from those chunks.

Two chunking approaches are compared, as the write-up asks:

| | Chunking | Metadata |
|---|---|---|
| **Sliding window** | All pages joined into one text, then split into 512-character chunks with 100 overlap, as in the course code | Each chunk is mapped back to the page(s) its characters came from |
| **Single page** | Each page split on its own with the same splitter, so no chunk ever crosses a page | Taken directly from its page |

**One fix to the course code.** It gives *every* chunk `doc_type: "Packaging Specification"`. With that, a `doc_type` filter keeps everything and removes nothing, and answers about certificates or declarations would be attributed to the packaging specification. Here each chunk gets the type of the page it actually came from.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## Step 0: Install dependencies

In [ ]:
%pip install -q llama-cpp-python==0.3.19 --only-binary=llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
%pip install -q pymupdf langchain-text-splitters llama-index-core llama-index-llms-llama-cpp llama-index-embeddings-huggingface huggingface_hub

## Step 1: Model and PDF

In [1]:
import os, json, re, time
import llama_cpp

MODEL_PATH = os.environ.get("MISTRAL_GGUF", "/content/models/mistral-7b-instruct-v0.2.Q4_K_M.gguf")
if not os.path.exists(MODEL_PATH):
    from huggingface_hub import hf_hub_download
    hf_hub_download("TheBloke/Mistral-7B-Instruct-v0.2-GGUF", "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
                    local_dir=os.path.dirname(MODEL_PATH))

PDF_PATH = os.environ.get("BLOB_PDF", "/content/pharma-blob-sample.pdf")
if not os.path.exists(PDF_PATH):
    import urllib.request
    urllib.request.urlretrieve(
        "https://drive.google.com/uc?export=download&id=1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw", PDF_PATH)
SOURCE_FILE = os.path.basename(PDF_PATH)
N_GPU_LAYERS = -1 if llama_cpp.llama_supports_gpu_offload() else 0

## Step 2: Page-level metadata

The page-to-document map is the output of the segmentation pipeline from Step 1 of this project (10/10 correct on this file). `doc_id` is a readable name such as `certificate_of_quality_02`, numbered in order of appearance, so the two certificates get different IDs.

In [2]:
import fitz  # PyMuPDF

SEGMENTATION = [  # from the segmentation pipeline (Step 1, final task)
    {"page": 0, "is_new_doc": "Yes", "doc_type": "Cover Letter", "page_in_doc": 0},
    {"page": 1, "is_new_doc": "Yes", "doc_type": "Certificate of Quality", "page_in_doc": 0},
    {"page": 2, "is_new_doc": "Yes", "doc_type": "Certificate of Quality", "page_in_doc": 0},
    {"page": 3, "is_new_doc": "Yes", "doc_type": "Packaging Specification", "page_in_doc": 0},
    {"page": 4, "is_new_doc": "No", "doc_type": "Packaging Specification", "page_in_doc": 1},
    {"page": 5, "is_new_doc": "Yes", "doc_type": "BSE/TSE Declaration", "page_in_doc": 0},
    {"page": 6, "is_new_doc": "Yes", "doc_type": "Material Description", "page_in_doc": 0},
    {"page": 7, "is_new_doc": "Yes", "doc_type": "Supplier Qualification", "page_in_doc": 0},
    {"page": 8, "is_new_doc": "No", "doc_type": "Supplier Qualification", "page_in_doc": 1},
    {"page": 9, "is_new_doc": "Yes", "doc_type": "Chain of Custody", "page_in_doc": 0},
]

with fitz.open(PDF_PATH) as pdf:
    texts = [page.get_text() for page in pdf]

pages, counters, doc_id = [], {}, None
for seg, text in zip(SEGMENTATION, texts):
    slug = re.sub(r"[^a-z]+", "_", seg["doc_type"].lower()).strip("_")
    if seg["is_new_doc"] == "Yes":
        counters[slug] = counters.get(slug, 0) + 1
        doc_id = f"{slug}_{counters[slug]:02d}"
    pages.append({"page": seg["page"], "doc_type": seg["doc_type"], "text": text, "doc_id": doc_id,
                  "source_file": SOURCE_FILE, "page_in_doc": seg["page_in_doc"]})

import pandas as pd
pd.DataFrame([{k: v for k, v in p.items() if k != "text"} for p in pages]).set_index("page")

,doc_type,doc_id,source_file,page_in_doc
page,,,,
0,Cover Letter,cover_letter_01,pharma-blob-sample.pdf,0
1,Certificate of Quality,certificate_of_quality_01,pharma-blob-sample.pdf,0
2,Certificate of Quality,certificate_of_quality_02,pharma-blob-sample.pdf,0
3,Packaging Specification,packaging_specification_01,pharma-blob-sample.pdf,0
4,Packaging Specification,packaging_specification_01,pharma-blob-sample.pdf,1
5,BSE/TSE Declaration,bse_tse_declaration_01,pharma-blob-sample.pdf,0
6,Material Description,material_description_01,pharma-blob-sample.pdf,0
7,Supplier Qualification,supplier_qualification_01,pharma-blob-sample.pdf,0
8,Supplier Qualification,supplier_qualification_01,pharma-blob-sample.pdf,1


## Step 3: Chunking, two ways

**Sliding window (course):** pages are joined and split as one text. The course does not say how to join them, and it turns out to matter, so two versions are tested: pages joined by a **blank line** (`"\n\n"`, a paragraph break the splitter prefers to cut at) and by a **single newline** (`"\n"`, an ordinary line break). `add_start_index=True` makes the splitter record where each chunk starts, so every chunk can be mapped back to the page or pages its characters came from. A chunk that runs across a page boundary into a *different document* is a **cross-document chunk**: whatever single `doc_type` it is given, part of its text is mislabelled.

**Single page:** every page is split on its own with the same settings. Chunks never cross a page, so their metadata is always exact.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from llama_index.core import Document

splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100, add_start_index=True)
SEP = "\n\n"

def page_metadata(p, chunk_index, **extra):
    return {"doc_type": p["doc_type"], "chunk_index": chunk_index, "doc_id": p["doc_id"], "source_file": SOURCE_FILE,
            "page": p["page"], "page_in_doc": p["page_in_doc"], **extra}

def sliding_window_documents(sep):
    # PyMuPDF ends every page with a newline; strip it so the separator alone decides how pages are joined
    full_text, page_spans = "", []
    for p in pages:
        body = p["text"].strip()
        page_spans.append((len(full_text), len(full_text) + len(body)))
        full_text += body + sep
    docs = []
    for i, chunk in enumerate(splitter.create_documents([full_text])):
        start = chunk.metadata["start_index"]
        end = start + len(chunk.page_content)
        covered = [p for p, (s, e) in zip(pages, page_spans) if s < end and e > start]
        first = covered[0]  # the chunk is labelled with the page it starts on
        docs.append(Document(text=chunk.page_content, metadata=page_metadata(
            first, i, pages_covered=str([p["page"] for p in covered]),
            crosses_documents=len({p["doc_id"] for p in covered}) > 1)))
    return docs

def single_page_documents():
    docs, i = [], 0
    for p in pages:
        for chunk in splitter.split_text(p["text"]):
            docs.append(Document(text=chunk, metadata=page_metadata(p, i, pages_covered=str([p["page"]]),
                                                                    crosses_documents=False)))
            i += 1
    return docs

CHUNKINGS = {"Sliding, blank-line join": sliding_window_documents("\n\n"),
             "Sliding, newline join": sliding_window_documents("\n"),
             "Single page": single_page_documents()}
for name, docs in CHUNKINGS.items():
    crossing = [d for d in docs if d.metadata["crosses_documents"]]
    print(f"{name:<25} {len(docs):>3} chunks, {len(crossing)} cross a document boundary")
print("\nCross-document chunks (newline join):")
for d in CHUNKINGS["Sliding, newline join"]:
    if d.metadata["crosses_documents"]:
        print(f"  chunk {d.metadata['chunk_index']:>2}: labelled {d.metadata['doc_type']!r}, pages {d.metadata['pages_covered']} "
              f"| ...{d.text[-60:].strip()!r}".replace(chr(10), " "))

Sliding, blank-line join   28 chunks, 0 cross a document boundary
Sliding, newline join      24 chunks, 8 cross a document boundary
Single page                28 chunks, 0 cross a document boundary

Cross-document chunks (newline join):
  chunk  2: labelled 'Cover Letter', pages [0, 1] | ...'ement system.\nIssued by Cytiva Westborough Quality Assurance'
  chunk  4: labelled 'Certificate of Quality', pages [1, 2] | ...'iance with our ISO 9001 certified quality management system.'
  chunk  7: labelled 'Certificate of Quality', pages [2, 3] | ...'on for AKTA ready Flow Kit assemblies, including primary and'
  chunk 10: labelled 'Packaging Specification', pages [4, 5] | ...'ompliance Statement)\nManufacturer:\nCytiva Sweden AB\nAddress:'
  chunk 13: labelled 'BSE/TSE Declaration', pages [5, 6] | ...'ts\nPart Number:\n29184612\nRevision:\nAD\n1. Product Description'
  chunk 16: labelled 'Material Description', pages [6, 7] | ...'November 2023\nAudit Result:\nApproved (no critical findings)'

## Step 4: Index both, with BGE-small

Bookkeeping fields stay out of the embedding; `doc_type` is kept in, because it describes the content.

In [4]:
from llama_index.core import Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    query_instruction="Represent this sentence for searching relevant passages: ")
Settings.llm = None
for docs in CHUNKINGS.values():
    for d in docs:
        d.excluded_embed_metadata_keys = [k for k in d.metadata if k != "doc_type"]

INDEXES = {name: VectorStoreIndex.from_documents(docs) for name, docs in CHUNKINGS.items()}

LLM is explicitly disabled. Using MockLLM.


## Step 5: Mistral for query routing and answers

- **Query routing** uses the same constrained-JSON classifier as the earlier routing task, with the labels written exactly as the chunk metadata stores them.
- **Answers** come from LlamaIndex's `get_response_synthesizer(response_mode=COMPACT)` with Mistral through `LlamaCPP`, in Mistral's `[INST] ... [/INST]` prompt format.

In [5]:
from llama_index.llms.llama_cpp import LlamaCPP

def completion_to_prompt(completion, system_prompt=None):
    system = (system_prompt.strip() + "\n\n") if system_prompt else ""
    return f"[INST] {system}{completion.strip()} [/INST]"

def messages_to_prompt(messages, system_prompt=None):
    text = "\n\n".join(m.content for m in messages if m.content)
    return completion_to_prompt(text, system_prompt)

llm = LlamaCPP(model_path=MODEL_PATH, temperature=0.0, max_new_tokens=256, context_window=4096,
               completion_to_prompt=completion_to_prompt, messages_to_prompt=messages_to_prompt,
               model_kwargs={"n_gpu_layers": N_GPU_LAYERS, "seed": 42}, verbose=False)

DOC_TYPES = ["Cover Letter", "Certificate of Quality", "Packaging Specification", "BSE/TSE Declaration",
             "Material Description", "Supplier Qualification", "Chain of Custody", "Unknown"]
TYPE_DEFINITIONS = '''- Cover Letter: a letter or email addressed to the reader, e.g. about storage, shipping or what a package contains
- Certificate of Quality: certifies one product lot or batch (lot numbers, manufacture/expiry dates, release tests)
- Packaging Specification: packaging materials, configuration, packaging part numbers, drawings, packaging change history
- BSE/TSE Declaration: statement that products contain no animal-origin materials (BSE/TSE), risk assessment
- Material Description: materials of construction, composition and physical properties of a product
- Supplier Qualification: supplier approval status, audits, certifications, sub-suppliers, quality agreement
- Chain of Custody: where assemblies are manufactured and how they move to the customer
- Unknown: the question does not point to one type'''

def classify_query_llm(query):
    schema = {"type": "object", "properties": {"doc_type": {"type": "string", "enum": DOC_TYPES}}, "required": ["doc_type"]}
    response = llm._model.create_chat_completion(
        messages=[{"role": "user", "content": f'''Which type of document in a pharmaceutical supplier bundle is most likely to answer this question?

Document types:
{TYPE_DEFINITIONS}

QUESTION: {query}

Return JSON with "doc_type".'''}],
        response_format={"type": "json_object", "schema": schema}, temperature=0.0, max_tokens=30)
    return json.loads(response["choices"][0]["message"]["content"])["doc_type"]

## Step 6: Questions

The course's question, plus four that each target a different document. Every question has a short answer string that must appear in the retrieved text for retrieval to count as successful.

Q5 tests the third write-up question, finding a **specific term**: it names one engineering change number (ECN) out of three in the change history.

In [6]:
QUESTIONS = [
    {"query": "Were there any packaging configuration changes?", "answer_doc": "packaging_specification_01", "must_contain": "PETG"},
    {"query": "Until when is the BSE/TSE risk assessment valid?", "answer_doc": "bse_tse_declaration_01", "must_contain": "15 March 2028"},
    {"query": "What is the expiration date of the High Flow Kit, lot 15102934?", "answer_doc": "certificate_of_quality_02", "must_contain": "20210712"},
    {"query": "Which sub-suppliers provide materials, and are they approved?", "answer_doc": "supplier_qualification_01", "must_contain": "Wacker"},
    {"query": "What change was made under ECN-2021-0541?", "answer_doc": "packaging_specification_01", "must_contain": "3mm"},
]
for q in QUESTIONS:
    q["predicted_doc_type"] = classify_query_llm(q["query"])
    print(f"{q['predicted_doc_type']:<26} <- {q['query']}")

Packaging Specification    <- Were there any packaging configuration changes?


BSE/TSE Declaration        <- Until when is the BSE/TSE risk assessment valid?


Certificate of Quality     <- What is the expiration date of the High Flow Kit, lot 15102934?


Supplier Qualification     <- Which sub-suppliers provide materials, and are they approved?


Supplier Qualification     <- What change was made under ECN-2021-0541?


## Step 7: Retrieval: two chunkings × filter on/off

`top_k=4`, as in the course. For each run:

- **On-target chunks:** retrieved chunks whose characters come from the answer document (judged by the pages they actually cover, not by their label).
- **Answer found:** whether the answer string appears in the retrieved text.

In [7]:
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

TOP_K = 4
doc_pages = {}
for p in pages:
    doc_pages.setdefault(p["doc_id"], set()).add(p["page"])

def retrieve(chunking, q, use_filter):
    kwargs = {"similarity_top_k": TOP_K}
    if use_filter and q["predicted_doc_type"] != "Unknown":
        kwargs["filters"] = MetadataFilters(filters=[
            MetadataFilter(key="doc_type", value=q["predicted_doc_type"], operator=FilterOperator.EQ)])
    return INDEXES[chunking].as_retriever(**kwargs).retrieve(q["query"])

def on_target(node, q):
    return bool(set(json.loads(node.metadata["pages_covered"])) & doc_pages[q["answer_doc"]])

rows = []
for chunking in CHUNKINGS:
    for use_filter in [False, True]:
        for q in QUESTIONS:
            nodes = retrieve(chunking, q, use_filter)
            rows.append({"Chunking": chunking, "Filter": "on" if use_filter else "off", "Question": q["query"][:45],
                         "On-target chunks": sum(on_target(n, q) for n in nodes),
                         "Answer found": q["must_contain"].lower() in " ".join(n.text for n in nodes).lower(),
                         "Cross-document chunks": sum(n.metadata["crosses_documents"] for n in nodes)})
retrieval = pd.DataFrame(rows)
summary = retrieval.groupby(["Chunking", "Filter"], sort=False).agg(
    on_target=("On-target chunks", "sum"), answers_found=("Answer found", "sum"),
    cross_document=("Cross-document chunks", "sum"))
summary["on_target"] = summary["on_target"].astype(str) + f" / {TOP_K * len(QUESTIONS)}"
summary["answers_found"] = summary["answers_found"].astype(str) + f" / {len(QUESTIONS)}"
summary

on_target answers_found  cross_document
Chunking                 Filter                                        
Sliding, blank-line join off      14 / 20         5 / 5               0
                         on       13 / 20         4 / 5               0
Sliding, newline join    off      14 / 20         5 / 5               7
                         on       13 / 20         4 / 5               6
Single page              off      14 / 20         5 / 5               0
                         on       13 / 20         4 / 5               0

On-target chunks (of 4) per question, single-page chunking:

In [8]:
retrieval[retrieval["Chunking"] == "Single page"].pivot_table(
    index="Question", columns="Filter", values="On-target chunks", sort=False)

Filter,off,on
Question,,
Were there any packaging configuration change,4.0,4.0
Until when is the BSE/TSE risk assessment val,3.0,3.0
What is the expiration date of the High Flow,2.0,2.0
"Which sub-suppliers provide materials, and ar",3.0,4.0
What change was made under ECN-2021-0541?,2.0,0.0


## Step 8: Final answers (single-page chunks + metadata filter + exact-term fallback)

Single-page chunks with the `doc_type` filter, plus one guard prompted by the results above. A wrong routing decision is invisible to the filter: it simply searches the wrong document. So **if the question names a specific identifier** (an ECN, document or lot number) **and none of the filtered chunks contain it, the search is repeated without the filter**. Identifiers are matched exactly, so this cannot be fooled by similar-sounding text.

Every answer is generated by Mistral from the four chunks retrieved, and packaged in the task's format.

In [9]:
from llama_index.core.response_synthesizers import get_response_synthesizer, ResponseMode

synthesizer = get_response_synthesizer(llm=llm, response_mode=ResponseMode.COMPACT)

def identifiers_in(query):
    # e.g. ECN-2021-0541, PKG-SPEC-2023-0847, 15102934
    return re.findall(r"\b(?:[A-Z]{2,}-[A-Z0-9-]*\d[A-Z0-9-]*|\d{6,})\b", query)

def retrieve_with_fallback(q):
    nodes = retrieve("Single page", q, use_filter=True)
    ids = identifiers_in(q["query"])
    text = " ".join(n.text for n in nodes)
    if ids and not any(i in text for i in ids):
        return retrieve("Single page", q, use_filter=False), True
    return nodes, False

final_outputs = []
for q in QUESTIONS:
    nodes, fallback = retrieve_with_fallback(q)
    t0 = time.perf_counter()
    answer = str(synthesizer.synthesize(q["query"], nodes=nodes)).strip()
    keep = ["doc_type", "chunk_index", "doc_id", "source_file", "page", "page_in_doc"]
    final_outputs.append({
        "query": q["query"], "predicted_doc_type": q["predicted_doc_type"], "filter_fallback_used": fallback,
        "matched_chunks": [{"text": n.text, "metadata": {k: n.metadata[k] for k in keep}} for n in nodes],
        "answer": answer,
    })
    print(f"\nQ: {q['query']}\n   routed to: {q['predicted_doc_type']}{' -> identifier not found, searched unfiltered' if fallback else ''}"
          f" | chunks from {sorted({n.metadata['doc_id'] for n in nodes})} | {time.perf_counter() - t0:.0f}s\nA: {answer}")

with open("contract_routing_results.json", "w") as f:
    json.dump(final_outputs, f, indent=2)


Q: Were there any packaging configuration changes?
   routed to: Packaging Specification | chunks from ['packaging_specification_01'] | 35s
A: Yes, based on the context information provided, there have been packaging configuration changes. Specifically, the blister material for the AKTA ready Flow Kit Packaging Assembly has been changed from PVC to PETG, and there have been updates to the tray dimensions, lid film sealing parameters, and label placement on the drawings. These changes were approved by Packaging Engineering at Cytiva Uppsala on November 8, 2023.



Q: Until when is the BSE/TSE risk assessment valid?
   routed to: BSE/TSE Declaration | chunks from ['bse_tse_declaration_01'] | 18s
A: The BSE/TSE risk assessment is valid until 15 March 2028.



Q: What is the expiration date of the High Flow Kit, lot 15102934?
   routed to: Certificate of Quality | chunks from ['certificate_of_quality_01', 'certificate_of_quality_02'] | 24s
A: The expiration date of the High Flow Kit, lot 15102934 is 20210712.



Q: Which sub-suppliers provide materials, and are they approved?
   routed to: Supplier Qualification | chunks from ['supplier_qualification_01'] | 43s
A: Based on the context information provided, the following sub-suppliers provide materials and have been approved:

1. DuPont (Tyvek) - Lid film (1073B) - Approved (2020)
2. Wacker Chemie AG - Silicone tubing - Approved (2019)
3. Sabic Innovative - Polypropylene resin - Approved (2021)

Therefore, the answer to the query is that DuPont (Tyvek), Wacker Chemie AG, and Sabic Innovative are approved sub-suppliers who provide materials.



Q: What change was made under ECN-2021-0541?
   routed to: Supplier Qualification -> identifier not found, searched unfiltered | chunks from ['bse_tse_declaration_01', 'packaging_specification_01'] | 30s
A: The change made under ECN-2021-0541 was an update to the tray insert dimensions. The dimensions were increased by +3mm based on customer feedback.


In [10]:
example = final_outputs[0]
print(json.dumps({**example, "matched_chunks": [{**c, "text": c["text"][:100] + "..."} for c in example["matched_chunks"]]}, indent=2))

{
  "query": "Were there any packaging configuration changes?",
  "predicted_doc_type": "Packaging Specification",
  "filter_fallback_used": false,
  "matched_chunks": [
    {
      "text": "5. Drawing Changes\nDrawing DWG-29477427-PKG-R3 supersedes DWG-29477427-PKG-R2.\nKey changes located o...",
      "metadata": {
        "doc_type": "Packaging Specification",
        "chunk_index": 12,
        "doc_id": "packaging_specification_01",
        "source_file": "pharma-blob-sample.pdf",
        "page": 4,
        "page_in_doc": 1
      }
    },
    {
      "text": "Packaging Component Specification (continued)\nDocument Number: PKG-SPEC-2023-0847, Page 2 of 2\n4. Co...",
      "metadata": {
        "doc_type": "Packaging Specification",
        "chunk_index": 11,
        "doc_id": "packaging_specification_01",
        "source_file": "pharma-blob-sample.pdf",
        "page": 4,
        "page_in_doc": 1
      }
    },
    {
      "text": "Packaging Component Specification\nDocument Number:

## Findings

Run with Mistral-7B on a laptop CPU, `top_k=4`, five questions.

**1. Chunking: single-page chunks are the safe choice; the sliding window only works by luck of formatting.**

| Chunking | Chunks | Chunks crossing into another document | Answers found (filter off / on) |
|---|---|---|---|
| Sliding window, pages joined by a blank line | 28 | 0 | 5/5 / 4/5 |
| Sliding window, pages joined by a single newline | 24 | **8** | 5/5 / 4/5 |
| Single page | 28 | 0 | 5/5 / 4/5 |

The course's sliding window behaved exactly like single-page chunking *only* when pages were joined with a blank line. `RecursiveCharacterTextSplitter` cuts at blank lines first, so every chunk ended at a page edge. Joined with an ordinary line break, a third of the chunks ran from one document into the next. For example, a chunk labelled "BSE/TSE Declaration" also contains the Material Description's part number, and two chunks labelled "Supplier Qualification" contain the start of the Chain of Custody list. On these five questions it did not change which answers were found, but those mislabelled chunks were returned 6 times *with* the filter on. That is text from the wrong document passing a filter that is supposed to exclude it. Splitting each page separately makes this impossible regardless of formatting.

**2. Metadata filtering helped little on this small file, and failed badly once.**
- *Sub-suppliers:* on-target chunks went from 3 to 4 of 4.
- *Packaging changes, BSE/TSE validity, certificate expiry:* no change, because plain vector search already found the right document.
- *Certificate expiry:* noise was **not** removed. 2 of the 4 chunks came from the *other* Certificate of Quality, because `doc_type` cannot tell two documents of the same type apart. Filtering on `doc_id` would.
- *"What change was made under ECN-2021-0541?":* the router sent this to Supplier Qualification, the filter removed every packaging chunk, and the answer could not be found (2 → 0 on-target). To its credit, Mistral then said it could not find the change rather than inventing one.

**3. An exact-term fallback fixed the routing failure.** If a question names an identifier (an ECN, document or lot number) and none of the filtered chunks contain it, the search is repeated without the filter. That recovered the ECN question: "tray insert dimensions increased by +3mm", which is correct. All five final answers are correct.

**4. Generation is a separate source of error.** For the sub-supplier question, all four sub-suppliers were in the retrieved chunks, but Mistral's answer named only three (it left out Schott AG). Good retrieval does not guarantee a complete answer.

**Fixes to the course code:**
- every chunk now gets the `doc_type` of the page it came from (the course labels everything "Packaging Specification", which makes the filter meaningless);
- `langchain.text_splitter` no longer exists, so the splitter is imported from `langchain_text_splitters`;
- a local LLM is set explicitly for the response synthesizer (by default LlamaIndex would try to call OpenAI).